# 05. Evaluate The MLX-SFT Student

This notebook loads the MLX-LM adapter from notebook 04 and evaluates it on the held-out τ³-Bench retail test split.

The benchmark loop is the same as notebook 01: the student agent talks to the simulated user and the retail environment, and the environment computes the reward.


In [ ]:
from __future__ import annotations

from collections import Counter
import json

from pathlib import Path
import sys

cwd = Path.cwd()
ROOT = cwd if (cwd / "common" / "config.py").exists() else cwd.parent
if not (ROOT / "common" / "config.py").exists():
    raise RuntimeError("Run this notebook from the repo root or a blog folder under the repo root.")
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

from common import config as cfg
from common import retail_eval
from common import tau_runtime
from common import user_simulator

paths = cfg.setup_notebook_paths(blog_dir_name="1-distilling-a-0-8b-tool-calling-agent")
ROOT = paths.root
BLOG_DIR = paths.blog_dir
DATA_DIR = paths.data_dir
OUTPUT_DIR = paths.output_dir
ENV_PATH = paths.env_path
DOTENV_LOADED = paths.dotenv_loaded

TAU_BENCH_REPO_REVISION = cfg.TAU_BENCH_REPO_REVISION
STUDENT_MODEL = cfg.STUDENT_MODEL
STUDENT_SLUG = cfg.filename_slug(STUDENT_MODEL)
SFT_WORK_DIR = OUTPUT_DIR / f"{STUDENT_SLUG}_tau3_retail_sft_mlx_lm"
ADAPTER_DIR = SFT_WORK_DIR / f"{STUDENT_SLUG}_tau3_retail_mlx_lora_adapter"

print("Project root:", ROOT)
print("Loaded .env:", DOTENV_LOADED, "from", ENV_PATH)
print("Student MLX base:", STUDENT_MODEL)
print("Adapter dir:", ADAPTER_DIR)
print("Adapter exists:", (ADAPTER_DIR / "adapters.safetensors").exists())

## Load The MLX Student Adapter


In [ ]:
if not (ADAPTER_DIR / "adapters.safetensors").exists():
    raise FileNotFoundError("Run notebook 04 first; no MLX adapter found at " + str(ADAPTER_DIR))

from mlx_lm import generate as mlx_generate
from mlx_lm import load as mlx_load
from mlx_lm.sample_utils import make_sampler

mlx_model, mlx_tokenizer = mlx_load(STUDENT_MODEL, adapter_path=str(ADAPTER_DIR))
mlx_sampler = make_sampler(temp=0.0, top_p=1.0, top_k=0)
print("Loaded MLX base plus adapter.")


## Shared MLX Student Runner

The baseline student eval and the SFT student eval now use the same MLX-LM agent and runner from the shared eval modules. This keeps the harness boundary identical before and after training; only the adapter changes.


## Load τ³-Bench Retail Test Tasks And User Simulator


In [ ]:
user_simulator_runtime = user_simulator.start_tau_bench_user_simulator_from_env(
    existing_shim=globals().get("chatgpt_user_simulator_shim")
)
chatgpt_user_simulator_shim = user_simulator_runtime.shim
TAU_BENCH_USER_SIMULATOR_LLM = user_simulator_runtime.model
TAU_BENCH_USER_SIMULATOR_ARGS = user_simulator_runtime.args

retail_domain = tau_runtime.load_tau_bench_retail_domain(DATA_DIR, TAU_BENCH_REPO_REVISION)
tau_bench_runtime = retail_domain.runtime
retail_test_tasks = tau_bench_runtime.get_tasks("test")
retail_environment = retail_domain.environment
retail_tools = retail_domain.tools
retail_tool_schema_by_name = retail_domain.tool_schema_by_name

print("Held-out retail tasks:", len(retail_test_tasks))
print("User simulator:", TAU_BENCH_USER_SIMULATOR_LLM)
print("User simulator args:", user_simulator.public_user_simulator_args(TAU_BENCH_USER_SIMULATOR_ARGS))


## Run Held-Out Eval


In [ ]:
TAU_BENCH_SFT_EVAL_LIMIT = len(retail_test_tasks)
TAU_BENCH_SFT_EVAL_MAX_STEPS = 100
TAU_BENCH_SFT_EVAL_MAX_ERRORS = 10
TAU_BENCH_SFT_EVAL_MAX_NEW_TOKENS = cfg.MAX_NEW_TOKENS
TAU_BENCH_SFT_EVAL_SEED = 42
TAU_BENCH_MLFLOW_ENABLED = True
TAU_BENCH_MLFLOW_EXPERIMENT_NAME = "distillation-blogs-tau3"
TAU_BENCH_MLFLOW_LOG_FULL_ARTIFACTS = True

user_simulator_slug = cfg.filename_slug(TAU_BENCH_USER_SIMULATOR_LLM)
sft_eval_output_path = SFT_WORK_DIR / f"{STUDENT_SLUG}_tau3_retail_mlx_sft_eval_{user_simulator_slug}.json"
sft_eval_trace_dir = OUTPUT_DIR / "local_traces" / f"{STUDENT_SLUG}_tau3_retail_mlx_sft_eval_{user_simulator_slug}"
sft_eval_mlflow_run_name = f"{STUDENT_SLUG}_tau3_retail_mlx_sft_eval_{user_simulator_slug}"
sft_eval_mlflow_config = cfg.MlflowConfig(
    enabled=TAU_BENCH_MLFLOW_ENABLED,
    experiment_name=TAU_BENCH_MLFLOW_EXPERIMENT_NAME,
    log_full_artifacts=TAU_BENCH_MLFLOW_LOG_FULL_ARTIFACTS,
    log_spans=False,
)

sft_eval_config = cfg.TauBenchRetailEvalConfig(
    dataset_revision=TAU_BENCH_REPO_REVISION,
    student_model_name=f"{STUDENT_MODEL} + {ADAPTER_DIR.name}",
    user_simulator_model=TAU_BENCH_USER_SIMULATOR_LLM,
    user_simulator_args=TAU_BENCH_USER_SIMULATOR_ARGS,
    max_steps=TAU_BENCH_SFT_EVAL_MAX_STEPS,
    max_errors=TAU_BENCH_SFT_EVAL_MAX_ERRORS,
    max_new_tokens=TAU_BENCH_SFT_EVAL_MAX_NEW_TOKENS,
    seed=TAU_BENCH_SFT_EVAL_SEED,
    model_role="mlx_sft_student",
)

sft_eval_runner = retail_eval.TauBenchRetailMlxStudentEvalRunner(
    runtime=tau_bench_runtime,
    model=mlx_model,
    tokenizer=mlx_tokenizer,
    qwen_tools=retail_tools,
    tool_schema_by_name=retail_tool_schema_by_name,
    sampler=mlx_sampler,
    config=sft_eval_config,
    trace_dir=sft_eval_trace_dir,
)

print("Official τ³-Bench retail SFT-student eval")
print("Student base:", STUDENT_MODEL)
print("Adapter dir:", ADAPTER_DIR)
print("User simulator model:", TAU_BENCH_USER_SIMULATOR_LLM)
print("Held-out retail tasks:", min(TAU_BENCH_SFT_EVAL_LIMIT, len(retail_test_tasks)))
print("Max steps per task:", TAU_BENCH_SFT_EVAL_MAX_STEPS)
print("Max tool errors per task:", TAU_BENCH_SFT_EVAL_MAX_ERRORS)
print("Max new tokens per agent action:", TAU_BENCH_SFT_EVAL_MAX_NEW_TOKENS)
print("Output path:", sft_eval_output_path)
print("Trace dir:", sft_eval_trace_dir)
print("MLflow enabled:", sft_eval_mlflow_config.enabled)

sft_eval_payload = retail_eval.run_tau_bench_retail_eval_tasks(
    task_objects=retail_test_tasks[:TAU_BENCH_SFT_EVAL_LIMIT],
    runner=sft_eval_runner,
    output_path=sft_eval_output_path,
    print_progress=True,
    show_progress_bar=True,
    quiet_tau2_console=True,
    mlflow_config=sft_eval_mlflow_config,
    mlflow_run_name=sft_eval_mlflow_run_name,
    mlflow_tags={"tau3.notebook": "05_eval_sft_student", "tau3.runtime": "mlx_lm_sft"},
)

sft_eval_rows = sft_eval_payload["task_results"]
sft_eval_correct = sum(1 for row in sft_eval_rows if row["is_success"])
sft_eval_accuracy = sft_eval_correct / len(sft_eval_rows) if sft_eval_rows else 0.0

print()
print("MLX-SFT student official τ³-Bench retail pass@1 accuracy:", round(sft_eval_accuracy, 4))
print(f"Correct: {sft_eval_correct}/{len(sft_eval_rows)}")
print("Saved aggregate results to:", sft_eval_output_path)
print("Saved per-task traces to:", sft_eval_trace_dir)
if sft_eval_mlflow_config.enabled:
    print("MLflow run:", sft_eval_mlflow_run_name)
    print("MLflow experiment:", sft_eval_mlflow_config.experiment_name)

failure_counter = Counter(
    row["termination_reason"] if not row.get("error") else f"error:{row['error']['type']}"
    for row in sft_eval_rows
    if not row["is_success"]
)
print()
print("Failure/termination summary:")
for name, count in failure_counter.most_common():
    print(f"  {name}: {count}")